# Siteora Image Model — Evaluation

Compares the base model against checkpoints and the final Siteora LoRA on a fixed evaluation prompt set (`src/evaluation/evaluate.py::EVAL_PROMPTS`).

In [ ]:
import sys
sys.path.insert(0, 'src')


## 1. Load base model (no LoRA) and generate baseline images

In [ ]:
from evaluation.evaluate import run_evaluation, EVAL_PROMPTS

BASE_MODEL = 'stabilityai/stable-diffusion-xl-base-1.0'
run_evaluation('base', 'evaluations', base_model=BASE_MODEL, lora_path=None)


## 2. Load Siteora LoRA checkpoints and generate for each

In [ ]:
import glob

checkpoint_dirs = sorted(glob.glob('checkpoints/standard/step-*'))
for i, ckpt in enumerate(checkpoint_dirs, 1):
    # Note: checkpoints store raw LoRA state dicts (src/training/checkpointing.py),
    # not a diffusers-loadable adapter dir; export via unet.save_lora_adapter first
    # if you want to evaluate an intermediate checkpoint directly.
    print('would evaluate', ckpt)


## 3. Run fixed prompts against the final exported LoRA

In [ ]:
run_evaluation('final', 'evaluations', base_model=BASE_MODEL, lora_path='models/siteora-lora-standard')


## 4. Compare checkpoints — simple report

In [ ]:
from evaluation.evaluate import build_comparison_report
report = build_comparison_report('evaluations', labels=['base', 'final'])
print(report)
open('evaluations/report.md', 'w').write(report)


## 5. Side-by-side visual comparison

In [ ]:
import json
from PIL import Image
import matplotlib.pyplot as plt

with open('evaluations/base/manifest.json') as f:
    base_manifest = json.load(f)

fig, axes = plt.subplots(2, len(EVAL_PROMPTS), figsize=(4*len(EVAL_PROMPTS), 8))
for i, item in enumerate(base_manifest['results']):
    base_img = Image.open(f"evaluations/base/{item['file_name']}")
    final_img = Image.open(f"evaluations/final/{item['file_name']}")
    axes[0, i].imshow(base_img); axes[0, i].axis('off'); axes[0, i].set_title(item['id'], fontsize=8)
    axes[1, i].imshow(final_img); axes[1, i].axis('off')
axes[0, 0].set_ylabel('base'); axes[1, 0].set_ylabel('+ siteora lora')
plt.tight_layout(); plt.show()
